# Pipeline Flow Analysis: Article Counts and Losses

This notebook tracks the number of articles at each stage of the UAIR pipeline and visualizes the flow using a Sankey diagram.


In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Set base paths for US and global pipeline outputs (symlinked)
BASE_PATH_US = Path('/share/pierson/matt/UAIR/outputs/for_nov10workshop_us_results')
BASE_PATH_GLOBAL = Path('/share/pierson/matt/UAIR/outputs/for_nov10workshop_global_results')

print(f'US base path: {BASE_PATH_US}')
print(f'US path exists: {BASE_PATH_US.exists()}')
print(f'Global base path: {BASE_PATH_GLOBAL}')
print(f'Global path exists: {BASE_PATH_GLOBAL.exists()}')


US base path: /share/pierson/matt/UAIR/outputs/for_nov10workshop_us_results
US path exists: True
Global base path: /share/pierson/matt/UAIR/outputs/for_nov10workshop_global_results
Global path exists: True


In [4]:
# Load data from each stage - combine US and global datasets
stages_data = {}

# Stage 1: Classification (all articles)
classify_all_path_us = BASE_PATH_US / 'classify' / 'classify_all.parquet'
classify_all_path_global = BASE_PATH_GLOBAL / 'classify' / 'classify_all.parquet'

df_classify_all_us = pd.read_parquet(classify_all_path_us) if classify_all_path_us.exists() else pd.DataFrame()
df_classify_all_global = pd.read_parquet(classify_all_path_global) if classify_all_path_global.exists() else pd.DataFrame()

df_classify_all = pd.concat([df_classify_all_us, df_classify_all_global], ignore_index=True)
stages_data['classify_all'] = df_classify_all
print(f'Stage 1 (Classify All): {len(df_classify_all)} articles (US: {len(df_classify_all_us)}, Global: {len(df_classify_all_global)})')

# Stage 1b: Classification (relevant only)
classify_relevant_path_us = BASE_PATH_US / 'classify' / 'classify_relevant.parquet'
classify_relevant_path_global = BASE_PATH_GLOBAL / 'classify' / 'classify_relevant.parquet'

df_classify_relevant_us = pd.read_parquet(classify_relevant_path_us) if classify_relevant_path_us.exists() else pd.DataFrame()
df_classify_relevant_global = pd.read_parquet(classify_relevant_path_global) if classify_relevant_path_global.exists() else pd.DataFrame()

df_classify_relevant = pd.concat([df_classify_relevant_us, df_classify_relevant_global], ignore_index=True)
stages_data['classify_relevant'] = df_classify_relevant
print(f'Stage 1b (Classify Relevant): {len(df_classify_relevant)} articles')
# Verify filtering logic
if 'is_relevant' in df_classify_all.columns:
    relevant_count = df_classify_all['is_relevant'].sum()
    print(f'  Verified: {relevant_count} articles have is_relevant=True')


Stage 1 (Classify All): 415821 articles (US: 29372, Global: 386449)
Stage 1b (Classify Relevant): 8609 articles
  Verified: 8609 articles have is_relevant=True


In [ ]:
# write classify joint to parquet 

print(df_classify_all.info())

JOINT_OUTPUT_PATH = Path('/share/pierson/matt/UAIR/outputs/for_nov10workshop_joint_results')
JOINT_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
df_classify_all.to_parquet(JOINT_OUTPUT_PATH / 'classify_all.parquet')


<bound method DataFrame.info of                                       article_id  \
0       a86f8be9d122f56af054170f36a9c673b817fc9b   
1       86eabb01c013ca66ff6c5f33c2cb2b86498d4034   
2       25e1051ad5ddfdd82da03d22429f436c04a66613   
3       0be7ff45c4428e39c8b488f0fabc258c51b3821c   
4       9f3b3875a586a124bfbee43e5672d6fe3101bff8   
...                                          ...   
415816  fcbf656e4159c276ecdb62ad33b73b373ea07a10   
415817  ae28fde74930e64b7b046295b8e4d9c0317a23af   
415818  51a953eb431a5882b247e22bd744db47d66a08d1   
415819  2aa3488a95a5727443b951d977589c6d3cb82bf7   
415820  9293bf808f09d03e32109d400f7a30ac733fab58   

                                             article_path  \
0       /share/pierson/matt/UAIR/data/us/2016/http:\\w...   
1       /share/pierson/matt/UAIR/data/us/2016/https:\\...   
2       /share/pierson/matt/UAIR/data/us/2016/http:\\w...   
3       /share/pierson/matt/UAIR/data/us/2016/http:\\u...   
4       /share/pierson/matt/UAIR/data/

In [6]:
# Stage 2: Decomposition
decompose_path_us = BASE_PATH_US / 'decompose_nbl' / 'tuples.parquet'
decompose_path_global = BASE_PATH_GLOBAL / 'decompose_nbl' / 'tuples.parquet'

df_decompose_us = pd.read_parquet(decompose_path_us) if decompose_path_us.exists() else pd.DataFrame()
df_decompose_global = pd.read_parquet(decompose_path_global) if decompose_path_global.exists() else pd.DataFrame()

df_decompose = pd.concat([df_decompose_us, df_decompose_global], ignore_index=True)
stages_data['decompose'] = df_decompose
print(f'Stage 2 (Decompose): {len(df_decompose)} articles')


Stage 2 (Decompose): 8609 articles


In [7]:
# Stage 3: Verification
verify_path_us = BASE_PATH_US / 'verify_nbl' / 'verify_nbl_results.parquet'
verify_path_global = BASE_PATH_GLOBAL / 'verify_nbl' / 'verify_nbl_results.parquet'

df_verify_us = pd.read_parquet(verify_path_us) if verify_path_us.exists() else pd.DataFrame()
df_verify_global = pd.read_parquet(verify_path_global) if verify_path_global.exists() else pd.DataFrame()

df_verify = pd.concat([df_verify_us, df_verify_global], ignore_index=True)
stages_data['verify'] = df_verify
print(f'Stage 3 (Verify): {len(df_verify)} articles')

# Check verification filtering
if 'core_tuple_verified' in df_verify.columns:
    verified_count = df_verify['core_tuple_verified'].sum()
    print(f'  Core tuple verified: {verified_count} articles')
    print(f'  Not verified: {len(df_verify) - verified_count} articles')


Stage 3 (Verify): 8609 articles
  Core tuple verified: 544 articles
  Not verified: 8065 articles


In [8]:
# Stage 4: EU AI Act Classification
eu_act_path_us = BASE_PATH_US / 'classify_eu_ai_act' / 'classify_eu_ai_act_results.parquet'
eu_act_path_global = BASE_PATH_GLOBAL / 'classify_eu_ai_act' / 'classify_eu_ai_act_results.parquet'

df_eu_act_us = pd.read_parquet(eu_act_path_us) if eu_act_path_us.exists() else pd.DataFrame()
df_eu_act_global = pd.read_parquet(eu_act_path_global) if eu_act_path_global.exists() else pd.DataFrame()

df_eu_act = pd.concat([df_eu_act_us, df_eu_act_global], ignore_index=True)
stages_data['eu_act'] = df_eu_act
print(f'Stage 4 (EU AI Act): {len(df_eu_act)} articles')

# Check vague article filtering
if 'too_vague_to_process' in df_eu_act.columns:
    vague_count = df_eu_act['too_vague_to_process'].sum()
    print(f'  Too vague to process: {vague_count} articles')
    print(f'  Successfully classified: {len(df_eu_act) - vague_count} articles')


Stage 4 (EU AI Act): 544 articles
  Too vague to process: 160 articles
  Successfully classified: 384 articles


In [9]:
# Stage 5: Risk and Benefits Classification
risk_benefits_path_us = BASE_PATH_US / 'classify_risk_benefits' / 'classify_risk_benefits_results.parquet'
risk_benefits_path_global = BASE_PATH_GLOBAL / 'classify_risk_benefits' / 'classify_risk_benefits_results.parquet'

df_risk_benefits_us = pd.read_parquet(risk_benefits_path_us) if risk_benefits_path_us.exists() else pd.DataFrame()
df_risk_benefits_global = pd.read_parquet(risk_benefits_path_global) if risk_benefits_path_global.exists() else pd.DataFrame()

df_risk_benefits = pd.concat([df_risk_benefits_us, df_risk_benefits_global], ignore_index=True)
stages_data['risk_benefits'] = df_risk_benefits
print(f'Stage 5 (Risk/Benefits): {len(df_risk_benefits)} articles')


Stage 5 (Risk/Benefits): 544 articles


## Calculate Article Counts and Losses


In [10]:
# Calculate counts at each stage
flow_data = {
    'stage': [],
    'count': [],
    'description': []
}

# Stage 0: Input
input_count = len(df_classify_all)
flow_data['stage'].append('Input')
flow_data['count'].append(input_count)
flow_data['description'].append('Raw articles')

# Stage 1: After classification (relevant)
relevant_count = len(df_classify_relevant)
flow_data['stage'].append('Classify')
flow_data['count'].append(relevant_count)
flow_data['description'].append('Relevant articles')

# Stage 2: After decomposition
decompose_count = len(df_decompose)
flow_data['stage'].append('Decompose')
flow_data['count'].append(decompose_count)
flow_data['description'].append('Decomposed tuples')

# Stage 3: After verification
verify_count = len(df_verify)
verified_count = df_verify['core_tuple_verified'].sum() if 'core_tuple_verified' in df_verify.columns else verify_count
flow_data['stage'].append('Verify')
flow_data['count'].append(verified_count)
flow_data['description'].append('Verified tuples')

# Stage 4: EU AI Act
eu_act_count = len(df_eu_act)
flow_data['stage'].append('EU AI Act')
flow_data['count'].append(eu_act_count)
flow_data['description'].append('EU Act classified')

# Stage 5: Risk/Benefits
risk_benefits_count = len(df_risk_benefits)
flow_data['stage'].append('Risk/Benefits')
flow_data['count'].append(risk_benefits_count)
flow_data['description'].append('Risk/Benefits classified')

# Create DataFrame
df_flow = pd.DataFrame(flow_data)
print('\nPipeline Flow Summary:')
print(df_flow.to_string(index=False))

# Calculate losses
print('\nLosses at each stage:')
for i in range(1, len(df_flow)):
    prev_count = df_flow.iloc[i-1]['count']
    curr_count = df_flow.iloc[i]['count']
    loss = prev_count - curr_count
    loss_pct = (loss / prev_count * 100) if prev_count > 0 else 0
    print(f"{df_flow.iloc[i-1]['stage']} → {df_flow.iloc[i]['stage']}: -{loss:,} articles ({loss_pct:.1f}% loss)")



Pipeline Flow Summary:
        stage  count              description
        Input 415821             Raw articles
     Classify   8609        Relevant articles
    Decompose   8609        Decomposed tuples
       Verify    544          Verified tuples
    EU AI Act    544        EU Act classified
Risk/Benefits    544 Risk/Benefits classified

Losses at each stage:
Input → Classify: -407,212 articles (97.9% loss)
Classify → Decompose: -0 articles (0.0% loss)
Decompose → Verify: -8,065 articles (93.7% loss)
Verify → EU AI Act: -0 articles (0.0% loss)
EU AI Act → Risk/Benefits: -0 articles (0.0% loss)


## Create Sankey Diagram


In [11]:

import plotly.graph_objects as go
import plotly.offline as pyo



In [12]:
# Prepare data for Sankey diagram
# Nodes: stages
stages_list = ['Raw Articles', 'Classify AI Relevant', 'AI Use Case Tuple Decomposition', 'Claim Verification', 'EU AI Act Classification', 'Risk/Benefits Classification']

# Counts at each stage
counts = [
    input_count,
    relevant_count,
    decompose_count,
    verified_count,
    eu_act_count,
    risk_benefits_count
]

# Create node labels with counts

# Helper function to format numbers to 2 significant digits
import math
def format_sig_figs(num, sig_figs=2):
    """Format number to specified significant digits"""
    if num == 0:
        return "0"
    magnitude = math.floor(math.log10(abs(num)))
    rounded = round(num, sig_figs - magnitude - 1)
    if abs(rounded) >= 1000:
        return f"{rounded:,.0f}" if rounded == int(rounded) else f"{rounded:,.{max(0, sig_figs - magnitude - 1)}f}"
    else:
        return f"{rounded:.{max(0, sig_figs - magnitude - 1)}f}"

node_labels = [f"{stage}\n({format_sig_figs(count)})" for stage, count in zip(stages_list, counts)]

# Define flows (source → target)
# Each flow represents articles passing from one stage to the next
source_nodes = []
target_nodes = []
flow_values = []
flow_labels = []

# Flow 1: Input → Classify (relevant articles pass through)
source_nodes.append(0)  # Input
target_nodes.append(1)  # Classify
flow_values.append(input_count)
flow_labels.append(f'{format_sig_figs(input_count)} raw articles')

# Flow 2: Classify → Decompose (all relevant articles decomposed)
source_nodes.append(1)  # Classify
target_nodes.append(2)  # Decompose
flow_values.append(relevant_count)
flow_labels.append(f'{format_sig_figs(relevant_count)} relevant articles')

# Flow 3: Decompose → Verify (verified articles pass through)
source_nodes.append(2)  # Decompose
target_nodes.append(3)  # Verify
flow_values.append(relevant_count)
flow_labels.append(f'{format_sig_figs(relevant_count)} decomposed use cases')

# Flow 4: Verify → EU AI Act (all verified articles classified)
source_nodes.append(3)  # Verify
target_nodes.append(4)  # EU AI Act
flow_values.append(eu_act_count)
flow_labels.append(f'{format_sig_figs(eu_act_count)} classified')

# Flow 5: EU AI Act → Risk/Benefits (all EU Act articles get risk/benefits)
source_nodes.append(3)  # Verify
target_nodes.append(5)  # Risk/Benefits
flow_values.append(risk_benefits_count)
flow_labels.append(f'{format_sig_figs(risk_benefits_count)} classified')

# Add loss flows (articles dropped)
# Loss 1: Input → Classify (non-relevant articles)
loss_1 = input_count - relevant_count
if loss_1 > 0:
    source_nodes.append(1)  # classify
    target_nodes.append(len(stages_list))  # Loss node
    flow_values.append(loss_1)
    flow_labels.append(f'{format_sig_figs(loss_1)} not relevant')

# Loss 2: Decompose → Verify (non-verified articles)
loss_2 = decompose_count - verified_count
if loss_2 > 0:
    source_nodes.append(3)  # verify
    target_nodes.append(len(stages_list))  # Loss node
    flow_values.append(loss_2)
    flow_labels.append(f'{format_sig_figs(loss_2)}  unverified')

# Add loss node to stages list
stages_list_with_loss = stages_list + ['Filtered Out']
node_labels_with_loss = node_labels + [f"Filtered \n({format_sig_figs(sum([loss_1, loss_2]))})"]

print(f'Total flows: {len(source_nodes)}')

# Import numpy for log transformation
import numpy as np

# Ensure flow_values is defined
if 'flow_values' not in locals() or len(flow_values) == 0:
    raise ValueError('flow_values is not defined. Please run the entire cell from the beginning.')

# Transform flow values using log scale for better visualization
# Keep original values for labels, but use log-transformed values for visualization
import numpy as np

# Store original flow values
# Store original flow values (with safety check)
try:
    flow_values_original = flow_values.copy()
except NameError:
    raise NameError('flow_values is not defined. Please run the entire cell from the beginning, starting from "# Prepare data for Sankey diagram"')

# Calculate percentages for labels
def get_percentage(value, source_idx):
    """Calculate percentage of flow relative to source node"""
    if source_idx == 0:  # Input node
        return (value / input_count * 100) if input_count > 0 else 0
    elif source_idx == 1:  # Classify node
        return (value / relevant_count * 100) if relevant_count > 0 else 0
    elif source_idx == 2:  # Decompose node
        return (value / decompose_count * 100) if decompose_count > 0 else 0
    elif source_idx == 3:  # Verify node
        return (value / verified_count * 100) if verified_count > 0 else 0
    elif source_idx == 4:  # EU AI Act node
        return (value / eu_act_count * 100) if eu_act_count > 0 else 0
    else:
        return 0

# Update flow labels to include percentages
flow_labels_with_pct = []
for i, (val, label, src) in enumerate(zip(flow_values_original, flow_labels, source_nodes)):
    pct = get_percentage(val, src)
    if pct > 0:
        flow_labels_with_pct.append(f'{label}\n({pct:.1f}%)')
    else:
        flow_labels_with_pct.append(label)


Total flows: 7


In [ ]:
# Create Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=15,  # Reduced thickness for better visibility of small flows
        line=dict(color="black", width=0.5),
        label=node_labels_with_loss,
        color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
    ),
    link=dict(
        # Note: Plotly Sankey automatically calculates flow curves.
        # Individual curve directions cannot be directly controlled.
        # The layout algorithm determines optimal paths.
        source=source_nodes,
        target=target_nodes,
        value=flow_values,
        label=flow_labels,
        color=["rgba(31, 119, 180, 1)" if t < len(stages_list) else "rgba(200, 0, 0, 0.6)"  # Darker red for Lost flows
               for t in target_nodes]
    )
)])

fig.update_layout(
    title_text="UAIR Pipeline Flow: Article Counts and Losses (Not to Scale)",
    font_size=18,
    width=1600,
    height=900
)

fig.show()


In [ ]:
# Save figure
output_dir = Path('/share/pierson/matt/UAIR/papers/uair_brief/figures')
output_dir.mkdir(parents=True, exist_ok=True)

# Save as HTML
html_path = output_dir / 'pipeline_flow_sankey.html'
fig.write_html(str(html_path))
print(f'Saved HTML to {html_path}')

# Save as PDF (requires kaleido)
try:
    pdf_path = output_dir / 'pipeline_flow_sankey.pdf'
    fig.write_image(str(pdf_path), width=1200, height=600, scale=2)
    print(f'Saved PDF to {pdf_path}')
    # Save as PNG
    png_path = output_dir / 'pipeline_flow_sankey.png'
    fig.write_image(str(png_path), width=1200, height=600, scale=2)
    print(f'Saved PNG to {png_path}')
except Exception as e:
    print(f'Could not save PDF (may need to install kaleido): {e}')


Saved HTML to /share/pierson/matt/UAIR/papers/uair_brief/figures/pipeline_flow_sankey.html


Saved PDF to /share/pierson/matt/UAIR/papers/uair_brief/figures/pipeline_flow_sankey.pdf
Saved PNG to /share/pierson/matt/UAIR/papers/uair_brief/figures/pipeline_flow_sankey.png


## Summary Statistics Table


In [ ]:
# Create a summary table
summary_data = {
    'Stage': stages_list,
    'Article Count': counts,
    'Cumulative Loss': [0] + [input_count - c for c in counts[1:]],
    'Stage Loss': [0] + [counts[i-1] - counts[i] for i in range(1, len(counts))],
    'Retention Rate (%)': [100.0] + [c / counts[0] * 100 for c in counts[1:]]
}

df_summary = pd.DataFrame(summary_data)
df_summary['Stage Loss'] = df_summary['Stage Loss'].apply(lambda x: f'{x:,}' if x > 0 else '-')
df_summary['Cumulative Loss'] = df_summary['Cumulative Loss'].apply(lambda x: f'{x:,}')
df_summary['Article Count'] = df_summary['Article Count'].apply(lambda x: f'{x:,}')
df_summary['Retention Rate (%)'] = df_summary['Retention Rate (%)'].apply(lambda x: f'{x:.2f}%')

print('\nPipeline Flow Summary Table:')
print(df_summary.to_string(index=False))

# Save to CSV
csv_path = Path('/share/pierson/matt/UAIR/papers/uair_brief/tables/pipeline_flow_summary.csv')
csv_path.parent.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(csv_path, index=False)
print(f'\nSaved summary table to {csv_path}')



Pipeline Flow Summary Table:
                          Stage Article Count Cumulative Loss Stage Loss Retention Rate (%)
                   Raw Articles       415,821               0          -            100.00%
           Classify AI Relevant         8,609         407,212    407,212              2.07%
AI Use Case Tuple Decomposition         8,609         407,212          -              2.07%
             Claim Verification           544         415,277      8,065              0.13%
       EU AI Act Classification           544         415,277          -              0.13%
   Risk/Benefits Classification           544         415,277          -              0.13%

Saved summary table to /share/pierson/matt/UAIR/papers/uair_brief/tables/pipeline_flow_summary.csv
